# Fine-tune the Vietnamese humanizer (QLoRA, T4-class)

Train on **Colab or Kaggle free GPU (T4/P100, 16 GB)**. Five cells, run top to bottom:

1. GPU check + clone repo + install deps
2. Point at the human-written corpus
3. Build the SFT dataset (`prepare_dataset.py`)
4. Train QLoRA (`train_qlora.py`; auto-falls back to 7B if the 14B OOMs)
5. Merge the adapter + optional AWQ export; persist artifacts

Edit the CONFIG cell once. Regenerate this notebook by editing
`training/remote/mk_notebook.py` — the notebook itself is disposable.

In [ ]:
# ==== 0. CONFIG ====
REPO_URL = "https://github.com/datnguyentien2006/vietnamese-humanize.git"
DATA_RAW = None            # Colab: r"/content/drive/MyDrive/vietnamese-humanize/data-raw" (a folder of .txt/.txt.gz/.jsonl/.parquet corpora)
                           # Kaggle: leave None and attach ONE dataset with the text corpus — auto-detected
PERSIST_DIR = None         # Colab: r"/content/drive/MyDrive/vietnamese-humanize/out" ; on Kaggle outputs are kept automatically (kernel output)
TRAIN_MODEL = None         # None -> use training/config.yaml (14B base). Set 'Qwen/Qwen2.5-7B-Instruct' to force fallback.
LIMIT = None               # None -> all corpus docs; e.g. 20000 for a quick dry run
RUN_AWQ_EXPORT = True

In [ ]:
# ==== 1. GPU check + clone + install ====
!nvidia-smi

import os, subprocess

if not os.path.isdir("vietnamese-humanize/.git"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "vietnamese-humanize"], check=True)
%cd vietnamese-humanize

!pip install -q -r training/requirements.txt
!pip install -q huggingface_hub
print("setup ok")

Wed Oct 21 11:37:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
import os
try:
    d = PERSIST_DIR
except NameError:
    d = '/content/drive/MyDrive/vietnamese-humanize-out'
print('kernel_alive lora=', os.path.isdir('training/output/humanizer-lora'))
print('awq_dir=', os.listdir('training/output/humanizer-awq')[:8] if os.path.isdir('training/output/humanizer-awq') else 'MISSING')
print('drive_out=', os.listdir(d)[:8] if os.path.isdir(d) else 'NO_DRIVE_DIR')
import glob
print('gpu=', bool(glob.glob('/dev/nvidia0')))

kernel_alive lora= False
awq_dir= MISSING
drive_out= NO_DRIVE_DIR
gpu= False


In [ ]:
# ==== 3. Build SFT dataset ====
limit_arg = "" if LIMIT is None else f"--limit {LIMIT}"
!python training/prepare_dataset.py {limit_arg}
with open("training/data/sft/train.jsonl", encoding="utf-8") as fh:
    print(fh.read(700))

In [ ]:
# ==== 4. Train QLoRA (14B, falls back to 7B on OOM) ====
import subprocess

FALLBACK = "Qwen/Qwen2.5-7B-Instruct"


def try_train(model_id):
    cmd = ["python", "training/train_qlora.py"]
    if model_id:
        cmd += ["--model", model_id]
    print("RUN:", *cmd)
    proc = subprocess.run(cmd)
    print("exit code:", proc.returncode)
    return proc.returncode == 0


ok = try_train(TRAIN_MODEL)
if not ok and TRAIN_MODEL != FALLBACK:
    print("[warn] falling back to", FALLBACK)
    ok = try_train(FALLBACK)
assert ok, "Training failed on both 14B and 7B — lower LIMIT or free disk/VRAM, then retry."
print("adapter: training/output/humanizer-lora")

In [ ]:
# ==== 5. Merge + AWQ export + persist ====
import os, shutil

if RUN_AWQ_EXPORT:
    !python training/merge_and_export.py --export-awq
else:
    !python training/merge_and_export.py

if PERSIST_DIR:
    os.makedirs(PERSIST_DIR, exist_ok=True)
    for kind in ("lora", "merged", "awq"):
        src = f"training/output/humanizer-{kind}"
        dst = os.path.join(PERSIST_DIR, f"humanizer-{kind}")
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print("persisted:", dst)
else:
    print("artifacts remain in training/output/* for this session (download manually on Colab)")
print("DONE")